# Deepfake Audio Detection: SE-CNN-GRU Classifier
**Robust Artificial Speech Classifier Using Log-Mel Spectrograms and Channel Attention**

This notebook builds a high-performance deep learning model to classify speech recordings as **Genuine (Human)** or **Deepfake (AI-Generated)**.

### Pipeline Highlights
- **Features**: 128-band Log-Mel Spectrograms with dynamic normalization.
- **Model**: 2D Convolutional Neural Network with Squeeze-and-Excitation (SE) attention blocks and Bidirectional Gated Recurrent Units (GRU).
- **Regularization**: Spatial Dropout2D and Dense L2 regularizers.
- **Training**: Auto-balancing datasets, learning rate scheduler, and early stopping.

## 1. Google Colab Environment Setup

In [ ]:
# @title Kaggle Credentials Setup
import os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    KAGGLE_USERNAME = "" # @param {type:"string"}
    KAGGLE_KEY = "" # @param {type:"string"}
    
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
        os.environ['KAGGLE_KEY'] = KAGGLE_KEY
        
        !pip install -q kaggle librosa soundfile
        
        print("Downloading The Fake-or-Real Dataset...")
        !kaggle datasets download -d mohammedabdeldayem/the-fake-or-real-dataset --unzip -p /content/the-fake-or-real-dataset
    else:
        print("Credentials missing. Please fill in Kaggle credentials for Colab run.")

## 2. Imports & GPU Initialization

In [ ]:
import os, json, sys
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from collections import Counter
from tqdm import tqdm
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve
)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Input, Conv2D, BatchNormalization, Activation, MaxPooling2D,
    SpatialDropout2D, Reshape, Bidirectional, GRU, Dense, Dropout,
    GlobalAveragePooling2D, Multiply
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2

print("TensorFlow Version:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices('GPU'))

## 3. Environment Detection & Path Bindings

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    BASE_PATH = "/content/the-fake-or-real-dataset/for-norm/for-norm"
    WORK_DIR = "/content/working"
else:
    # Fallback to local / Kaggle directories
    BASE_PATH = "/kaggle/input/datasets/mohammedabdeldayem/the-fake-or-real-dataset/for-norm/for-norm"
    if not os.path.exists(BASE_PATH):
        BASE_PATH = "./the-fake-or-real-dataset/for-norm/for-norm"
    WORK_DIR = "./working"

FEAT_DIR = os.path.join(WORK_DIR, "feats")
MODEL_PATH = os.path.join(WORK_DIR, "best_model.keras")
CONFIG_PATH = os.path.join(WORK_DIR, "model_config.json")

os.makedirs(FEAT_DIR, exist_ok=True)
print(f"Active Dataset Path: {BASE_PATH} (Exists: {os.path.exists(BASE_PATH)})")
print(f"Outputs Directory  : {WORK_DIR}")

## 4. Pipeline Constants

In [ ]:
SR = 16000
DURATION = 4.0
N_MELS = 128
N_FFT = 512
HOP_LENGTH = 500  # Frame shift size (~31 ms overlap)
TARGET_FRAMES = 128
FEATURE_SHAPE = (128, 128, 1)
BATCH_SIZE = 32
LABEL_MAP = {"fake": 1, "real": 0}
SUBSET_LIMIT = None  # None for full training run; set to integer for fast testing

## 5. Exploratory Data Analysis & Visualizations

In [ ]:
# Plot side-by-side Genuine vs Deepfake signals to inspect physical changes
if os.path.exists(BASE_PATH):
    sample_real = os.path.join(BASE_PATH, "training", "real", os.listdir(os.path.join(BASE_PATH, "training", "real"))[0])
    sample_fake = os.path.join(BASE_PATH, "training", "fake", os.listdir(os.path.join(BASE_PATH, "training", "fake"))[0])

    def plot_sample(file_path, label):
        y, sr = librosa.load(file_path, sr=SR, duration=DURATION)
        plt.figure(figsize=(14, 4))
        
        # Waveform plot
        plt.subplot(1, 2, 1)
        librosa.display.waveshow(y, sr=sr, color='blue' if label == 'Genuine' else 'red')
        plt.title(f"{label} Waveform")
        plt.xlabel("Time (s)")
        plt.ylabel("Amplitude")
        
        # Mel-Spectrogram plot
        plt.subplot(1, 2, 2)
        S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
        S_dB = librosa.power_to_db(S, ref=np.max)
        librosa.display.specshow(S_dB, sr=sr, hop_length=HOP_LENGTH, x_axis='time', y_axis='mel')
        plt.colorbar(format='%+2.0f dB')
        plt.title(f"{label} Mel-Spectrogram")
        
        plt.tight_layout()
        plt.show()

    plot_sample(sample_real, "Genuine")
    plot_sample(sample_fake, "Deepfake")
else:
    print("Dataset path not found. Download the dataset to view sample waves.")

## 6. Spectrogram Extraction Pipeline

In [ ]:
def extract_features(file_path):
    """Extract Log-Mel Spectrogram normalized features (128, 128, 1)"""
    if isinstance(file_path, np.ndarray): file_path = file_path.item()
    if isinstance(file_path, bytes):      file_path = file_path.decode("utf-8")

    target_samples = int(SR * DURATION)
    try:
        y, _ = librosa.load(file_path, sr=SR, duration=DURATION)
    except Exception:
        y = np.zeros(target_samples, dtype=np.float32)

    # Padding & clipping to duration bounds
    y = np.pad(y, (0, max(0, target_samples - len(y))))[:target_samples]
    
    # Audio pre-emphasis to highpass filter signals
    y = np.append(y[0], y[1:] - 0.97 * y[:-1]).astype(np.float32)

    # Mel Spectrogram conversion
    S = librosa.feature.melspectrogram(y=y, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    S_dB = librosa.power_to_db(S, ref=np.max)

    # Frame dimension bounds alignment
    T = S_dB.shape[1]
    if T < TARGET_FRAMES:
        S_dB = np.pad(S_dB, ((0, 0), (0, TARGET_FRAMES - T)), mode='constant')
    else:
        S_dB = S_dB[:, :TARGET_FRAMES]

    # Feature standardization to zero-mean, unit-variance
    mean = S_dB.mean(axis=1, keepdims=True)
    std = S_dB.std(axis=1, keepdims=True) + 1e-6
    S_dB = (S_dB - mean) / std

    return S_dB.astype(np.float32)[..., np.newaxis]  # Shape: (128, 128, 1)

if os.path.exists(BASE_PATH):
    feature = extract_features(sample_real)
    print("Output Feature Tensor Shape:", feature.shape)
    assert feature.shape == FEATURE_SHAPE
    print("Pipeline Verification: Valid!")

## 7. Dataset Directory Parsing

In [ ]:
train_files, train_labels = [], []
val_files,   val_labels   = [], []
test_files,  test_labels  = [], []

SPLITS = {
    "training":   (train_files, train_labels),
    "validation": (val_files,   val_labels),
    "testing":    (test_files,  test_labels),
}

for split, (flist, llist) in SPLITS.items():
    for cls, label in LABEL_MAP.items():
        folder = os.path.join(BASE_PATH, split, cls)
        if not os.path.exists(folder): continue
        wavs = [os.path.join(folder, f) for f in os.listdir(folder) if f.endswith(".wav")]
        flist.extend(wavs)
        llist.extend([label] * len(wavs))

print("Train Set Sample Distribution:", len(train_files), Counter(train_labels))
print("Val Set Sample Distribution:  ", len(val_files),   Counter(val_labels))
print("Test Set Sample Distribution: ", len(test_files),  Counter(test_labels))

## 8. Cache Feature Arrays on Disk

In [ ]:
def precompute_split(files, labels, save_dir):
    os.makedirs(save_dir, exist_ok=True)
    
    if SUBSET_LIMIT is not None:
        files = files[:SUBSET_LIMIT]
        labels = labels[:SUBSET_LIMIT]

    npy_paths = []
    for i, (fp, _) in enumerate(tqdm(zip(files, labels), total=len(files))):
        out_path = os.path.join(save_dir, f"{i}.npy")
        if not os.path.exists(out_path):
            feat = extract_features(fp)
            np.save(out_path, feat)
        npy_paths.append(out_path)
    return npy_paths

if os.path.exists(BASE_PATH):
    print("Precomputing train set features...")
    train_npy = precompute_split(train_files, train_labels, os.path.join(FEAT_DIR, "train"))

    print("\nPrecomputing val set features...")
    val_npy = precompute_split(val_files, val_labels, os.path.join(FEAT_DIR, "val"))

    print("\nPrecomputing test set features...")
    test_npy = precompute_split(test_files, test_labels, os.path.join(FEAT_DIR, "test"))

    # Balance labels list match to subset limit bounds
    train_labels = train_labels[:len(train_npy)]
    val_labels = val_labels[:len(val_npy)]
    test_labels = test_labels[:len(test_npy)]

## 9. Construct TensorFlow Input Pipeline

In [ ]:
def load_npy(path, label):
    feat = tf.numpy_function(
        lambda p: np.load(p.decode()).astype(np.float32),
        [path], tf.float32
    )
    feat.set_shape(FEATURE_SHAPE)
    return feat, label

def spec_augment(features, label):
    H, W, C = FEATURE_SHAPE
    
    # Random frequency masking
    f0 = tf.random.uniform([], 0, 16, dtype=tf.int32)
    f = tf.random.uniform([], 0, H - f0, dtype=tf.int32)
    freq_mask = tf.concat([
        tf.ones([f, W, C]), tf.zeros([f0, W, C]), tf.ones([H - f - f0, W, C])
    ], axis=0)
    features = features * freq_mask

    # Random time masking
    t0 = tf.random.uniform([], 0, 16, dtype=tf.int32)
    t = tf.random.uniform([], 0, W - t0, dtype=tf.int32)
    time_mask = tf.concat([
        tf.ones([H, t, C]), tf.zeros([H, t0, C]), tf.ones([H, W - t - t0, C])
    ], axis=1)
    features = features * time_mask

    return features, label

def make_dataset(npy_files, labels, shuffle=False, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((npy_files, labels))
    if shuffle:
        ds = ds.shuffle(len(npy_files), seed=42)
    ds = ds.map(load_npy, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(spec_augment, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

if os.path.exists(BASE_PATH):
    train_ds = make_dataset(train_npy, train_labels, shuffle=True, augment=True)
    val_ds   = make_dataset(val_npy,   val_labels)
    test_ds  = make_dataset(test_npy,  test_labels)

## 10. SE-CNN-GRU Network Architecture

In [ ]:
def squeeze_excitation(inputs, ratio=8):
    """Channel-wise Squeeze-and-Excitation feature recalibration"""
    channels = inputs.shape[-1]
    squeeze = GlobalAveragePooling2D()(inputs)
    excitation = Dense(channels // ratio, activation='relu')(squeeze)
    excitation = Dense(channels, activation='sigmoid')(excitation)
    excitation = Reshape((1, 1, channels))(excitation)
    return Multiply()([inputs, excitation])

def conv_block(x, filters, pool):
    x = Conv2D(filters, (3, 3), padding="same", use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation("swish")(x)
    x = squeeze_excitation(x)
    x = MaxPooling2D(pool)(x)
    x = SpatialDropout2D(0.15)(x)
    return x

def build_model(input_shape=FEATURE_SHAPE):
    inputs = Input(shape=input_shape)
    
    # Feature compression layers
    x = conv_block(inputs, 32, (2, 2))  # (64, 64, 32)
    x = conv_block(x,      64, (2, 2))  # (32, 32, 64)
    x = conv_block(x,      128, (2, 2)) # (16, 16, 128)
    x = conv_block(x,      128, (2, 2)) # (8, 8, 128)
    
    # Sequence folding
    s = x.shape
    x = Reshape((s[1], s[2] * s[3]))(x) # (8, 1024)
    
    # Recurrent sequential model (cuDNN optimized with recurrent_dropout=0)
    x = Bidirectional(GRU(128, return_sequences=True, dropout=0.25))(x)
    x = Bidirectional(GRU(64, return_sequences=False, dropout=0.25))(x)
    
    # Classification head
    x = Dense(128, activation="swish", kernel_regularizer=l2(1e-3))(x)
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation="sigmoid")(x)
    
    return Model(inputs, outputs, name="AntiSpoof_SE_CNN_GRU")

model = build_model()
model.summary()

## 11. Compile & Model Configuration

In [ ]:
model.compile(
    optimizer=Adam(learning_rate=5e-4, clipnorm=1.0),
    loss=BinaryCrossentropy(label_smoothing=0.02),
    metrics=["accuracy"]
)
print("Model compiled successfully.")

## 12. Training Process Execution

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        mode="min",
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    )
]

if os.path.exists(BASE_PATH):
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=15,
        callbacks=callbacks
    )

## 13. Plot Learning Curves

In [ ]:
if os.path.exists(BASE_PATH) and 'history' in locals():
    epochs_range = range(1, len(history.history['loss']) + 1)
    
    plt.figure(figsize=(12, 4))
    
    # Loss Plot
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, history.history['loss'], label='Train Loss')
    plt.plot(epochs_range, history.history['val_loss'], label='Val Loss')
    plt.title('Loss Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    # Accuracy Plot
    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, history.history['accuracy'], label='Train Acc')
    plt.plot(epochs_range, history.history['val_accuracy'], label='Val Acc')
    plt.title('Accuracy Curves')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    
    plt.tight_layout()
    plt.show()

## 14. Validation Threshold Optimization (EER Minimization)

In [ ]:
if os.path.exists(BASE_PATH):
    best_model = load_model(MODEL_PATH)
    
    val_probs, val_true = [], []
    for X_batch, y_batch in val_ds:
        p = best_model.predict(X_batch, verbose=0)
        val_probs.extend(p.flatten())
        val_true.extend(y_batch.numpy())
    
    val_probs = np.array(val_probs)
    val_true = np.array(val_true)
    
    # Find EER threshold
    fpr, tpr, thresholds = roc_curve(val_true, val_probs)
    fnr = 1 - tpr
    eer_idx = np.argmin(np.abs(fpr - fnr))
    best_t = thresholds[eer_idx]
    
    # Safety clamp
    if best_t < 0.1 or best_t > 0.9:
        best_t = 0.5
        
    print(f"Optimized Validation Threshold (EER criteria): {best_t:.4f}")
    
    # Save config files
    with open(CONFIG_PATH, 'w') as f:
        json.dump({"threshold": float(best_t)}, f)
    print(f"Config saved to {CONFIG_PATH}")

## 15. Evaluation on Unseen Test Dataset

In [ ]:
if os.path.exists(BASE_PATH):
    test_probs, test_true = [], []
    for X_batch, y_batch in test_ds:
        p = best_model.predict(X_batch, verbose=0)
        test_probs.extend(p.flatten())
        test_true.extend(y_batch.numpy())
        
    test_probs = np.array(test_probs)
    test_true = np.array(test_true)
    
    # Apply optimized threshold
    test_pred = (test_probs > best_t).astype(int)
    
    overall_acc = accuracy_score(test_true, test_pred)
    f1 = f1_score(test_true, test_pred, zero_division=0)
    auc = roc_auc_score(test_true, test_probs)
    cm = confusion_matrix(test_true, test_pred)
    
    real_acc = cm[0, 0] / cm[0].sum() if cm[0].sum() > 0 else 0
    fake_acc = cm[1, 1] / cm[1].sum() if cm[1].sum() > 0 else 0
    
    fpr, tpr, _ = roc_curve(test_true, test_probs)
    fnr = 1 - tpr
    eer_idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
    
    print("=" * 54)
    print("            TEST EVALUATION RESULTS")
    print("=" * 54)
    metrics = [
        ("Overall Accuracy", overall_acc, 0.80, ">="),
        ("EER", eer, 0.12, "<="),
        ("F1 Score", f1, 0.80, ">="),
        ("Real Class Accuracy", real_acc, 0.75, ">="),
        ("Fake Class Accuracy", fake_acc, 0.75, ">="),
    ]
    all_pass = True
    for name, val, thresh, op in metrics:
        passed = (val >= thresh) if op == ">=" else (val <= thresh)
        if not passed: all_pass = False
        print(f"  {name:<26} {val:.4f}  (need {op}{thresh})  {'✅' if passed else '❌'}")
    print(f"  {'ROC AUC':<26} {auc:.4f}")
    print("=" * 54)
    print("  OVERALL STATUS:", "✅ ALL CRITERIA PASSED" if all_pass else "❌ SOME METRICS FAILED")
    print("=" * 54)
    print()
    print(classification_report(test_true, test_pred, target_names=["Real", "Fake"]))

## 16. Evaluation Plots (Confusion Matrix & ROC Curve)

In [ ]:
if os.path.exists(BASE_PATH) and 'test_pred' in locals():
    plt.figure(figsize=(12, 5))
    
    # Confusion Matrix Heatmap
    plt.subplot(1, 2, 1)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    plt.ylabel('True Labels')
    plt.xlabel('Predicted Labels')
    plt.title('Test Confusion Matrix')
    
    # ROC Curve Plot
    plt.subplot(1, 2, 2)
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    
    plt.tight_layout()
    plt.show()

## 17. Generate Inference Script (`predict.py`)

In [ ]:
%%writefile predict.py
"""
predict.py - Deepfake Audio Detection Inference
Usage: python predict.py path/to/audio.wav [model_path]
"""
import sys, json, os
import numpy as np
import librosa
from tensorflow.keras.models import load_model

SR, DURATION, N_FFT, HOP_LENGTH = 16000, 4.0, 512, 500
TARGET_FRAMES, N_MELS = 128, 128

def extract_features(file_path):
    n = int(SR * DURATION)
    try:
        audio, _ = librosa.load(file_path, sr=SR, duration=DURATION)
    except Exception:
        audio = np.zeros(n, dtype=np.float32)
    audio = np.pad(audio, (0, max(0, n - len(audio))))[:n]
    audio = np.append(audio[0], audio[1:] - 0.97 * audio[:-1]).astype(np.float32)
    
    S = librosa.feature.melspectrogram(y=audio, sr=SR, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    feat = librosa.power_to_db(S, ref=np.max)
    
    T = feat.shape[1]
    feat = np.pad(feat, ((0,0),(0, max(0, TARGET_FRAMES-T))))[:, :TARGET_FRAMES]
    feat = (feat - feat.mean(1, keepdims=True)) / (feat.std(1, keepdims=True) + 1e-6)
    return feat.astype(np.float32)[np.newaxis, ..., np.newaxis]

def predict_audio(file_path, model_path=None, config_path=None):
    if model_path is None:
        for p in ["best_model.keras", "working/best_model.keras", "/kaggle/working/best_model.keras", "/content/working/best_model.keras"]:
            if os.path.exists(p):
                model_path = p
                break
        if model_path is None:
            model_path = "best_model.keras"
            
    if config_path is None:
        for p in ["model_config.json", "working/model_config.json", "/kaggle/working/model_config.json", "/content/working/model_config.json"]:
            if os.path.exists(p):
                config_path = p
                break
        if config_path is None:
            config_path = "model_config.json"

    if not os.path.exists(model_path):
        print(f"Error: Model file not found at {model_path}")
        return None, 0.0

    model = load_model(model_path)
    thresh = 0.5
    if os.path.exists(config_path):
        try:
            with open(config_path) as f:
                cfg = json.load(f)
            thresh = cfg.get("threshold", 0.5)
        except Exception:
            pass

    prob   = model.predict(extract_features(file_path), verbose=0)[0][0]
    label  = "Deepfake (AI-Generated)" if prob > thresh else "Genuine (Human)"
    conf   = prob if prob > thresh else (1 - prob)
    print(f"File      : {file_path}")
    print(f"Result    : {label}")
    print(f"Confidence: {conf*100:.1f}%")
    print(f"Fake prob : {prob:.4f}  (threshold: {thresh:.2f})")
    return label, float(conf)

if __name__ == "__main__":
    if len(sys.argv) < 2:
        print("Usage: python predict.py path/to/audio.wav [model_path]")
    else:
        m_path = sys.argv[2] if len(sys.argv) > 2 else None
        predict_audio(sys.argv[1], model_path=m_path)
